# 04 Prophet Forecast

Prophet を fixed A/B の予測評価に接続する。今回は最小実装として、祝日効果なし・外生変数なしで評価する。

月次データなので `weekly_seasonality=False`, `daily_seasonality=False`, `yearly_seasonality=True` とする。Prophet は log scale の `y` で学習し、予測後に `exp(yhat)` で `number_parcels` の原系列スケールへ戻して評価する。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.evaluation import evaluate_forecasts
from src.forecasting.prophet_model import fit_prophet, forecast_prophet
from src.forecasting.splits import make_fixed_split_a, make_fixed_split_b


DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
PREDICTIONS_DIR = FORECAST_DIR / "predictions"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

for path in [PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PROPHET_SPEC = "prophet_no_regressors"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Forecast output:", FORECAST_DIR)

## 1. データ読み込みと固定分割

接続済みデータを読み込み、既存の fixed A/B split を使う。fixed B がデータ期間不足で作れない場合は理由を表示して skip する。

In [ ]:
df = load_connected_parcel_data(str(DATA_PATH))

splits = {}
split_rows = []
for split_name, maker in [("fixed_a", make_fixed_split_a), ("fixed_b", make_fixed_split_b)]:
    try:
        split = maker(df)
        splits[split_name] = split
        split_rows.append(
            {
                "split": split_name,
                "status": "success",
                "train_start": split["train"].index.min().date(),
                "train_end": split["train"].index.max().date(),
                "train_rows": len(split["train"]),
                "test_start": split["test"].index.min().date(),
                "test_end": split["test"].index.max().date(),
                "test_rows": len(split["test"]),
                "message": "",
            }
        )
    except ValueError as exc:
        split_rows.append(
            {
                "split": split_name,
                "status": "skipped",
                "train_start": None,
                "train_end": None,
                "train_rows": 0,
                "test_start": None,
                "test_end": None,
                "test_rows": 0,
                "message": str(exc),
            }
        )

split_summary = pd.DataFrame(split_rows)
display(split_summary)

## 2. Prophet without holidays / regressors

Prophet は `ds`, `y` の形式で学習する。ここでは `y=log(number_parcels)` を使い、future は `make_future_dataframe` ではなく test 期間の月次 index を明示的に渡す。

In [ ]:
prediction_frames = []
fit_rows = []

for split_name, split in splits.items():
    train = split["train"]
    test = split["test"]
    print(f"Fitting {split_name} Prophet")

    try:
        model = fit_prophet(train)
        forecast = forecast_prophet(model, test, split=split_name, spec_name=PROPHET_SPEC)
        prediction_frames.append(forecast)
        fit_rows.append(
            {
                "model": "prophet",
                "split": split_name,
                "forecast_type": "unconditional",
                "spec_name": PROPHET_SPEC,
                "yearly_seasonality": True,
                "weekly_seasonality": False,
                "daily_seasonality": False,
                "holidays": False,
                "regressors": "",
                "status": "success",
                "error": "",
            }
        )
    except Exception as exc:
        fit_rows.append(
            {
                "model": "prophet",
                "split": split_name,
                "forecast_type": "unconditional",
                "spec_name": PROPHET_SPEC,
                "yearly_seasonality": True,
                "weekly_seasonality": False,
                "daily_seasonality": False,
                "holidays": False,
                "regressors": "",
                "status": "failed",
                "error": str(exc),
            }
        )
        print(f"Prophet failed for {split_name}: {exc}")

if not prediction_frames:
    raise RuntimeError("No Prophet forecasts were created.")

predictions_df = pd.concat(prediction_frames, ignore_index=True)
fit_summary = pd.DataFrame(fit_rows)
display(fit_summary)

## 3. 評価指標と既存baselineとの比較

評価は `number_parcels` の原系列スケールで行う。既存の naive / SARIMA-SARIMAX / SSM metrics があれば読み込み、同じ表で比較する。

In [ ]:
metrics_frames = []
for split_name, split in splits.items():
    split_predictions = predictions_df[predictions_df["split"] == split_name]
    if split_predictions.empty:
        continue
    split_metrics = evaluate_forecasts(
        split_predictions,
        y_train=split["train"]["number_parcels"],
    )
    metrics_frames.append(split_metrics)

prophet_metrics = pd.concat(metrics_frames, ignore_index=True)

comparison_frames = []
for metrics_name in ["naive_metrics.csv", "sarimax_metrics.csv", "ssm_metrics.csv"]:
    metrics_path = METRICS_DIR / metrics_name
    if metrics_path.exists():
        comparison_frames.append(pd.read_csv(metrics_path))
comparison_frames.append(prophet_metrics)
comparison_metrics = pd.concat(comparison_frames, ignore_index=True)

display(prophet_metrics)
display(comparison_metrics.sort_values(["split", "rmse"]))

## 4. 予測結果と評価指標の保存

共通フォーマットの予測結果と評価指標を `output/forecasts/` 配下に保存する。論文用の `output/tables/` と `output/figures/` は使わない。

In [ ]:
if "fixed_a" in splits:
    predictions_df[predictions_df["split"] == "fixed_a"].to_csv(PREDICTIONS_DIR / "fixed_a_prophet.csv", index=False)

if "fixed_b" in splits:
    predictions_df[predictions_df["split"] == "fixed_b"].to_csv(PREDICTIONS_DIR / "fixed_b_prophet.csv", index=False)

prophet_metrics.to_csv(METRICS_DIR / "prophet_metrics.csv", index=False)

print("Saved Prophet prediction and metric files.")

## 5. 予測図の保存

テスト期間の実績値と Prophet forecast を比較する。学習期間の最後の24か月も薄く表示し、forecast の始点を確認しやすくする。

In [ ]:
def plot_split_forecast(split_name: str, save_path: Path) -> None:
    split = splits[split_name]
    plot_df = predictions_df[predictions_df["split"] == split_name]
    if plot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    train_tail = split["train"].tail(24)
    ax.plot(train_tail.index, train_tail["number_parcels"], color="0.55", linewidth=1.2, label="train actual tail")
    ax.plot(split["test"].index, split["test"]["number_parcels"], color="black", linewidth=1.6, label="test actual")
    ax.plot(plot_df["date"], plot_df["y_pred"], marker="o", linewidth=1.2, label="prophet")

    ax.axvline(split["train"].index.max(), color="0.2", linestyle=":", linewidth=1.0)
    ax.set_title(f"{split_name}: Prophet forecast comparison")
    ax.set_xlabel("Date")
    ax.set_ylabel("number_parcels")
    ax.grid(True, color="0.85", linewidth=0.8)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


if "fixed_a" in splits:
    plot_split_forecast("fixed_a", FIGURES_DIR / "fixed_a_prophet_forecast.png")

if "fixed_b" in splits:
    plot_split_forecast("fixed_b", FIGURES_DIR / "fixed_b_prophet_forecast.png")

print("Saved Prophet forecast figures.")

## 6. 設定メモ

この notebook の Prophet は holidays なし・regressors なしの unconditional baseline である。月次データなので週次・日次季節性は使わず、年次季節性とトレンドを中心に評価する。

後続で `baseline_m4` regressors を追加する場合は、SARIMAX/SSM と同じく train で定数の列を除外し、test期間のイベントダミーを既知として与える conditional forecast として別仕様に分ける。